In [ ]:
import os

from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

# ---- CONFIG ----
COLLECTION = "test_codebase"
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ---- Clients ----
openai = OpenAI(api_key=OPENAI_API_KEY)

qdrant = QdrantClient(url="http://localhost:6333")

neo4j_driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password123") 
)

In [ ]:
def classify_query(q: str) -> str:
    q = q.lower()

    if "where is" in q or "used" in q:
        return "reverse_lookup"

    if "how" in q or "flow" in q or "happens" in q:
        return "flow"

    return "semantic"

In [ ]:
def vector_search(question: str, top_k: int = 5):
    q_embedding = openai.embeddings.create(
        model=EMBED_MODEL,
        input=[question]
    ).data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_embedding,
        limit=top_k,
        with_payload=True,
    )

    return results.points or []

In [ ]:
def expand_graph(node_ids: list[str]):
    with neo4j_driver.session() as session:
        result = session.run("""
        MATCH (f:Function)
        WHERE f.id IN $node_ids

        OPTIONAL MATCH (f)-[:CALLS]->(called)
        OPTIONAL MATCH (caller)-[:CALLS]->(f)
        OPTIONAL MATCH (f)-[:BELONGS_TO]->(c:Class)
        OPTIONAL MATCH (f)-[:DEFINED_IN]->(file:File)

        RETURN f.id as id,
               collect(DISTINCT called.id) as callees,
               collect(DISTINCT caller.id) as callers,
               collect(DISTINCT c.name) as classes,
               collect(DISTINCT file.path) as files
        """, node_ids=node_ids)

        return [record.data() for record in result]

In [ ]:
def build_context(vector_results, graph_results):
    context = []

    context.append("PRIMARY FUNCTIONS:\n")

    for r in vector_results:
        p = r.payload
        context.append(
            f"{p['file']}:{p['start_line']}-{p['end_line']}\n"
            f"{p['name']}()\n"
            f"{p['source'][:500]}\n"
        )

    context.append("\nGRAPH RELATIONSHIPS:\n")

    for g in graph_results:
        context.append(
            f"Function: {g['id']}\n"
            f"Calls: {g['callees']}\n"
            f"Called By: {g['callers']}\n"
        )

    return "\n".join(context)

In [ ]:
def generate_answer(question: str, context: str):
    prompt = f"""
You are a senior software engineer analyzing a codebase.

Answer ONLY based on the provided context.
DO NOT hallucinate.

Provide:
1. Clear explanation
2. Execution flow (if applicable)
3. Exact file + line citations

---

CONTEXT:
{context}

---

QUESTION:
{question}
"""

    response = openai.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
def graph_reverse_lookup(question: str):
    """
    Find all functions that call a given function (reverse CALLS)
    Example: 'where is validate_email used?'
    """

    # --- crude keyword extraction (works for now) ---
    q = question.lower()

    if "where is" in q:
        keyword = q.split("where is")[-1]
    else:
        keyword = q

    keyword = keyword.replace("used", "").replace("?", "").strip()

    with neo4j_driver.session() as session:
        result = session.run("""
        MATCH (caller:Function)-[:CALLS]->(callee:Function)
        WHERE toLower(callee.name) CONTAINS toLower($keyword)

        OPTIONAL MATCH (caller)-[:DEFINED_IN]->(file:File)

        RETURN 
            callee.name as target,
            caller.name as caller,
            file.path as file
        """, keyword=keyword)

        rows = [r.data() for r in result]

    return rows

In [ ]:
def ask(question: str):
    print(f"\nQuestion: {question}")

    intent = classify_query(question)
    print(f"Intent: {intent}")

    # =========================
    # 🔥 1. REVERSE LOOKUP MODE
    # =========================
    if intent == "reverse_lookup":
        results = graph_reverse_lookup(question)

        if not results:
            print("\nNo usages found.")
            return

        print("\nANSWER:\n")
        print(f"Function usage results:\n")

        for r in results:
            print(f"- {r['caller']} → {r['target']} ({r['file']})")

        return

    # =========================
    # 🔍 2. VECTOR + GRAPH MODE
    # =========================
    vector_results = vector_search(question, top_k=5)

    if not vector_results:
        print("\nNo relevant functions found.")
        return

    node_ids = [r.payload["node_id"] for r in vector_results]

    graph_results = expand_graph(node_ids)

    context = build_context(vector_results, graph_results)

    answer = generate_answer(question, context)

    print("\nANSWER:\n")
    print(answer)

In [ ]:
questions = [
    "how is user creation handled?",
    "where is validate_email used?",
    "how does order placement work?",
    "what happens when user is deleted?",
]

for q in questions:
    ask(q)
    print("\n" + "="*80 + "\n")


Question: how is user creation handled?
Intent: flow

ANSWER:

1. **Clear Explanation:**
   User creation is handled by the `create_user` method in the `UserService` class located in `services/user_service.py`. This method takes three parameters: `username`, `email`, and an optional `role` (defaulting to "user"). The process involves validating the provided email using the `validate_email` function. If the email is invalid, a `ValueError` is raised. If the email is valid, a new `User` object is instantiated with the provided `username`, `email`, and `role`. This user object is then stored in the `users` dictionary of the `UserService` instance, and the `save` method of the `User` class is called to persist the user data. Finally, the newly created user object is returned.

2. **Execution Flow:**
   - The `create_user` method is called with the necessary parameters.
   - The email is validated using `validate_email(email)`.
   - If valid, a new `User` object is created by calling `User